# CaptionBench: one-notebook Colab baseline

Run the project from this notebook. You do not need to copy Python files or run a web server.
The setup cell installs the small helper package from the public repository.

Choose **Runtime > Change runtime type > GPU**. Run the cells in order.
The default is a 20-image validation pilot. Change the settings for the full test run after the pilot.

The notebook uses public local models. It makes no paid model API calls.
Colab GPU availability and runtime duration vary. Completed captions can persist in your Drive run folder.
See the [Colab FAQ](https://research.google.com/colaboratory/faq.html).

In [ ]:
# @title Install the project
import importlib.util
import json
import subprocess
import sys
from pathlib import Path

try:
    IN_COLAB = importlib.util.find_spec("google.colab") is not None
except ModuleNotFoundError:
    IN_COLAB = False
CODE_REF = "v0.1.0"
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    f"captionbench[models] @ git+https://github.com/jhunter11/cse434-fp.git@{CODE_REF}"], check=True)

import torch
from PIL import Image, ImageOps
from IPython.display import display, HTML
from captionbench.data import load_manifest, prepare_flickr8k, read_jsonl
from captionbench.models import caption_image, generate_to_file, release_models
from captionbench.evaluation import evaluate
from captionbench.ratings import create_rating_packet, write_rating_packet
def show_image(path, size=(600, 400)):
    with Image.open(path) as source:
        preview = ImageOps.exif_transpose(source).convert("RGB")
    preview.thumbnail(size)
    display(preview)

print("Project ready. No source files need manual copying.")

In [ ]:
# @title Study settings
USE_DRIVE = True # @param {type:"boolean"}
RUN_NAME = "validation-pilot-01" # @param {type:"string"}
MODEL_KEYS = "generic,blip-base" # @param {type:"string"}
RUN_SPLIT = "val" # @param ["val", "test"]
IMAGE_LIMIT = 20 # @param {type:"integer"}
DATA_ARCHIVE = "/content/drive/MyDrive/CaptionBench/flickr8k.zip" # @param {type:"string"}
DATA_ROOT = "/content/flickr8k" # @param {type:"string"}
RUN_SMOKE_IMAGE = True # @param {type:"boolean"}
RUN_INFERENCE = True # @param {type:"boolean"}

models = [name.strip() for name in MODEL_KEYS.split(",") if name.strip()]
allowed = {"generic", "blip-base", "blip-large", "blip2-opt-2.7b"}
assert models and len(models) == len(set(models)) and set(models) <= allowed, "Choose unique supported model names."
assert IMAGE_LIMIT > 0, "Choose a positive image count."
assert RUN_NAME and Path(RUN_NAME).name == RUN_NAME and RUN_NAME not in {".", ".."}, "Use one folder name."
assert RUN_SPLIT in {"val", "test"}

## Save results and check the GPU

Approve Drive mounting yourself if you want persistent results.
Keep the dataset archive in a project subfolder. The notebook extracts it into the runtime for faster image access.
Only run files and small result tables go to Drive. Rerun the same run name to resume an interrupted condition.
A different GPU, model revision, or library version requires a new run name.

In [ ]:
# @title Connect storage and inspect hardware
if IN_COLAB and USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    result_root = Path("/content/drive/MyDrive/CaptionBench/runs")
else:
    result_root = Path("/content/captionbench-runs") if IN_COLAB else Path.cwd() / "outputs" / "notebook-runs"
run = result_root / RUN_NAME
run.mkdir(parents=True, exist_ok=True)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
hardware = {"device": DEVICE, "torch": torch.__version__}
if DEVICE == "cuda":
    hardware.update(gpu=torch.cuda.get_device_name(0), vram_gib=round(torch.cuda.get_device_properties(0).total_memory / 2**30, 2))
else:
    print("No GPU is connected. Select a GPU runtime before the real model conditions.")
print(hardware)
print("Results:", run)

## Check one real caption

This optional cell uses Salesforce's public demonstration photo.
It checks inference and shows a caption directly in the notebook.
It is not a Flickr8k sample and has no reference-based benchmark score here.

In [ ]:
# @title Run the BLIP base smoke image
if RUN_SMOKE_IMAGE and DEVICE == "cuda":
    from urllib.request import urlretrieve
    smoke_dir = Path("/content/captionbench-smoke") if IN_COLAB else Path.cwd() / "data" / "smoke"
    smoke_dir.mkdir(parents=True, exist_ok=True)
    smoke_path = smoke_dir / "demo.jpg"
    if not smoke_path.exists():
        urlretrieve("https://storage.googleapis.com/sfr-vision-language-research/BLIP/demo.jpg", smoke_path)
    result = caption_image(smoke_path, model_key="blip-base", device=DEVICE)
    show_image(smoke_path)
    print(result["caption"])
    print("Inference seconds:", round(result["latency_seconds"], 3), "Revision:", result["revision"])
    (run / "smoke.json").write_text(json.dumps(result, indent=2), encoding="utf-8")
    release_models()
else:
    print("Smoke image skipped. GPU required by this notebook's smoke cell.")

## Prepare Flickr8k once

Acquire Flickr8k under the original source terms. Put its images, caption file, and original split lists in the configured archive or folder.
The importer finds the standard file names inside nested archive folders.
This repository does not redistribute the dataset.

Keep `RUN_SPLIT=val` for the feasibility pilot. After settings are fixed, use `test`, `IMAGE_LIMIT=1000`, and a new run name.
The full test condition requires the original 1,000 test IDs.

In [ ]:
# @title Extract local data and prepare the fixed split
from zipfile import ZipFile
root = Path(DATA_ROOT)
archive = Path(DATA_ARCHIVE)
if archive.is_file() and not any(root.rglob("Flickr_8k.testImages.txt")):
    root.mkdir(parents=True, exist_ok=True)
    with ZipFile(archive) as zipped:
        for member in zipped.infolist():
            destination = (root / member.filename).resolve()
            if not destination.is_relative_to(root.resolve()):
                raise ValueError("Archive contains an unsafe path.")
        zipped.extractall(root)

def unique_file(*names):
    matches = sorted({p for name in names for p in root.rglob(name) if p.is_file()})
    if len(matches) != 1:
        raise ValueError(f"Expected one of {names}, found {len(matches)}. Check DATA_ROOT.")
    return matches[0]

manifest = run / "flickr8k-manifest.jsonl"
records = []
if root.exists() and any(root.rglob("Flickr_8k.testImages.txt")):
    captions_file = unique_file("Flickr8k.token.txt", "captions.txt")
    split_paths = [unique_file(f"Flickr_8k.{part}Images.txt") for part in ("train", "dev", "test")]
    first_image = next(line.strip() for line in split_paths[2].read_text(encoding="utf-8-sig").splitlines() if line.strip())
    images_dir = unique_file(first_image).parent
    summary = prepare_flickr8k(images_dir, captions_file, *split_paths, manifest)
    records = load_manifest(manifest)
    assert summary["split_counts"]["test"] == 1000, "The original study needs exactly 1,000 test images."
    print(summary["split_counts"])
else:
    print("Dataset pending. Place the approved archive at DATA_ARCHIVE or extract it under DATA_ROOT, then rerun this cell.")

## Generate captions with saved progress

This cell runs one model at a time and releases GPU memory between models.
Each completed image appends to its prediction file. Resume skips verified completed images.
The runner rejects changed images or mixed model settings rather than merging incompatible results.
The generic control does not inspect image content.

In [ ]:
# @title Run the selected model conditions
selected = [row for row in records if row["split"] == RUN_SPLIT][:IMAGE_LIMIT]
predictions = []
can_infer = RUN_INFERENCE and (DEVICE == "cuda" or models == ["generic"])
if selected and can_infer:
    for model in models:
        print("Model:", model, "Images:", len(selected))
        try:
            rows = generate_to_file(selected, run / f"{model}-{RUN_SPLIT}.jsonl", model_key=model,
                                    split=RUN_SPLIT, device=DEVICE, resume=True)
            predictions.extend(rows)
        finally:
            release_models()
    print("Completed captions:", len(predictions))
elif selected:
    print("Inference skipped. Enable RUN_INFERENCE and connect a GPU for real models. Saved predictions can be scored below on CPU.")
else:
    print("Prepare the dataset first. No study predictions were generated.")


In [ ]:
# @title Score the shared test images
import pandas as pd
scores = None
predictions = []
if selected:
    paths = [run / f"{model}-{RUN_SPLIT}.jsonl" for model in models]
    if all(path.is_file() for path in paths):
        predictions = []
        selected_ids = {row["image_id"] for row in selected}
        for model, path in zip(models, paths):
            stored = read_jsonl(path)
            if any(row.get("model") != model for row in stored):
                raise ValueError("A saved prediction file contains the wrong model condition.")
            predictions.extend(row for row in stored if row["image_id"] in selected_ids)
if selected and RUN_SPLIT == "test" and predictions:
    if {row["model"] for row in predictions} != set(models):
        raise ValueError("Every selected model condition needs saved predictions before scoring.")
    scores = evaluate(selected, predictions)
    (run / "evaluation.json").write_text(json.dumps(scores, indent=2), encoding="utf-8")
    table = [{"model": model, "images": result["image_count"], **result["metrics"],
              "mean_inference_seconds": result["mean_latency_seconds"]}
             for model, result in scores["per_model"].items()]
    display(pd.DataFrame(table).set_index("model"))
    print("Null metrics are unmeasured. This table covers", len(selected), "test images.")
else:
    print("No test table yet. Every selected model needs saved test predictions. Validation pilots do not produce test metrics.")

In [ ]:
# @title View captions directly in the notebook
from html import escape
if selected and predictions:
    for row in selected[:8]:
        show_image(row["image_path"], (480, 320))
        matching = [p for p in predictions if p["image_id"] == row["image_id"]]
        text = "".join(f"<p><b>{escape(p['model'])}</b>: {escape(p['caption'])}</p>" for p in matching)
        refs = "".join(f"<li>{escape(ref)}</li>" for ref in row["references"])
        display(HTML(text + "<details><summary>Five human references</summary><ul>" + refs + "</ul></details>"))
else:
    print("No dataset gallery yet. The smoke cell above can still verify real inference.")

## Create the human rating packet

Use the same 100 test images for every model. Accuracy and detail remain separate ratings from 1 to 5.
Send raters only the form, rubric, and selected images with usable paths.
Keep the model key, predictions, scores, and gallery hidden until ratings finish.
A blank form is not a completed study.

In [ ]:
# @title Create a blinded 100-image form
if scores is not None and len(selected) >= 100:
    packet_dir = run / "ratings"
    if (packet_dir / "ratings.csv").exists():
        print("Existing rating packet preserved:", packet_dir)
    else:
        packet = create_rating_packet(selected, predictions, sample_size=100, seed=42)
        print(write_rating_packet(packet, packet_dir))
        print("Completed human ratings: 0. See selection.json for the rubric.")
else:
    print("Run at least 100 shared test images before creating the study form.")

In [ ]:
# @title Analyze completed human ratings on CPU
from captionbench.ratings import analyze_ratings, read_ratings_csv
completed_form = run / "ratings" / "completed-ratings.csv"
private_key_path = run / "ratings" / "private-model-key.json"
if scores is not None and completed_form.is_file() and private_key_path.is_file():
    key = json.loads(private_key_path.read_text(encoding="utf-8"))
    human = analyze_ratings(read_ratings_csv(completed_form), key, scores, bootstrap_samples=1000, seed=42)
    (run / "human-analysis.json").write_text(json.dumps(human, indent=2), encoding="utf-8")
    display(pd.DataFrame([{ "model": model, "mean_accuracy": row["mean_accuracy"], "mean_detail": row["mean_detail"],
                            "rated_captions": row["rated_caption_count"] } for model, row in human["per_model"].items()]))
    print("Separate metric correlations and image-cluster confidence intervals are in human-analysis.json.")
else:
    print("No completed human ratings yet. Combine filled forms into completed-ratings.csv with one header row.")


## Caption your own image inside Colab

This optional upload stays in your Colab session and uses the chosen local model.
It does not call a hosted vision API or start a tunneled web interface.

In [ ]:
# @title Optional single-image demo
if IN_COLAB:
    import io
    import ipywidgets as widgets
    from google.colab import files
    output = widgets.Output()
    model_picker = widgets.Dropdown(options=["blip-base", "blip-large", "generic"], value="blip-base", description="Model:")
    upload_button = widgets.Button(description="Upload and caption", button_style="success")
    def on_upload(_):
        upload_button.disabled = True
        with output:
            output.clear_output()
            try:
                uploaded = files.upload()
                if uploaded:
                    name, content = next(iter(uploaded.items()))
                    if len(content) > 10 * 1024 * 1024:
                        raise ValueError("Choose an image smaller than 10 MB.")
                    from tempfile import TemporaryDirectory
                    with TemporaryDirectory() as directory:
                        path = Path(directory) / "upload.png"
                        with Image.open(io.BytesIO(content)) as source:
                            from PIL import ImageOps
                            ImageOps.exif_transpose(source).convert("RGB").save(path)
                        result = caption_image(path, model_key=model_picker.value, device=DEVICE)
                        display(Image.open(path).copy())
                        print(result["caption"])
            except Exception as error:
                print(type(error).__name__ + ":", error)
            finally:
                release_models()
                upload_button.disabled = False
    upload_button.on_click(on_upload)
    display(model_picker, upload_button, output)
else:
    print("The upload widget runs in Colab. Use captionbench-web for the local browser demo.")

## Finish and share

Results persist under the printed run folder when Drive is enabled.
Clear notebook outputs before sharing any images, captions, or ratings that have restricted rights.
Disconnect the GPU after model inference. For CPU analysis, set RUN_INFERENCE=False and rerun setup, storage, data, and scoring cells with the same run name.

The [README](https://github.com/jhunter11/cse434-fp) describes local runs and completed-rating analysis.
The [research review](https://github.com/jhunter11/cse434-fp/blob/main/docs/research.md) explains prior methods and limitations.
Colab hardware and disconnect behavior come from the [official FAQ](https://research.google.com/colaboratory/faq.html).
This notebook has local code checks. A hosted Colab GPU run remains unverified until executed there.